In [1]:
pip install xgboost

In [2]:
pip install pandas numpy scikit-learn xgboost imbalanced-learn joblib matplotlib seaborn fastapi uvicorn

In [3]:
from google.colab import drive
drive.mount("/content/drive")

import zipfile
with zipfile.ZipFile("/content/drive/MyDrive/MachineLearningCSV.zip", "r") as z:
    z.extractall("/content/")

DATA_DIR = "/content/MachineLearningCVE"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
 import os

# Check what extracted
print(os.listdir("/content/"))
print("---")

# Look for the CSV folder
for root, dirs, files in os.walk("/content/"):
    for f in files:
        if f.endswith(".csv"):
            print(os.path.join(root, f))
    # Stop going too deep
    if root.count(os.sep) > 3:
        break

['.config', 'MachineLearningCVE', 'drive', 'sample_data']
---


In [5]:
from google.colab import drive
drive.mount("/content/drive")


import zipfile, os

zip_path = "/content/drive/MyDrive/MachineLearningCSV.zip"

# Confirm the zip is actually there
print("Zip exists:", os.path.exists(zip_path))

# Extract it
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall("/content/")
    print("Extracted files:")
    print(z.namelist()[:5])  # show first 5 entries

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Zip exists: True
Extracted files:
['MachineLearningCVE/', 'MachineLearningCVE/Wednesday-workingHours.pcap_ISCX.csv', 'MachineLearningCVE/Tuesday-WorkingHours.pcap_ISCX.csv', 'MachineLearningCVE/Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv', 'MachineLearningCVE/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv']


In [5]:
"""
Neural-Trace ML Pipeline
========================
Dataset : CIC-IDS-2017 (MachineLearningCVE CSVs)
Task    : Multi-class network intrusion classification + risk scoring
Output  : trained model (model.pkl), scaler (scaler.pkl), label encoder (label_encoder.pkl)

Run:
    pip install pandas numpy scikit-learn xgboost imbalanced-learn joblib
    python neural_trace_ml_pipeline.py
"""

import os
import glob
import warnings
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# CONFIG
# ─────────────────────────────────────────────
DATA_DIR        = "/content/MachineLearningCVE"
OUTPUT_DIR      = "./model_output"
MODEL_PATH      = f"{OUTPUT_DIR}/model.pkl"
SCALER_PATH     = f"{OUTPUT_DIR}/scaler.pkl"
ENCODER_PATH    = f"{OUTPUT_DIR}/label_encoder.pkl"
FEATURES_PATH   = f"{OUTPUT_DIR}/feature_list.pkl"

# Classes too rare to learn reliably — will be dropped
DROP_CLASSES = ["Infiltration", "Heartbleed", "Web Attack \ufffd Sql Injection"]

# Risk score mapping per attack type (used at inference time)
RISK_MAP = {
    "BENIGN":                      1,
    "Bot":                         8,
    "DDoS":                        9,
    "DoS GoldenEye":               8,
    "DoS Hulk":                    8,
    "DoS Slowhttptest":            7,
    "DoS slowloris":               7,
    "FTP-Patator":                 6,
    "PortScan":                    5,
    "SSH-Patator":                 6,
    "Web Attack \ufffd Brute Force": 7,
    "Web Attack \ufffd XSS":         7,
}

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ─────────────────────────────────────────────
# STEP 1: LOAD & MERGE ALL CSVs
# ─────────────────────────────────────────────
def load_data(data_dir):
    print("\n[1/6] Loading CSVs...")
    csv_files = glob.glob(os.path.join(data_dir, "*.csv"))
    dfs = []
    for f in sorted(csv_files):
        print(f"  Loading: {os.path.basename(f)}")
        df = pd.read_csv(f, low_memory=False)
        df.columns = df.columns.str.strip()      # remove leading/trailing spaces
        dfs.append(df)
    combined = pd.concat(dfs, ignore_index=True)
    print(f"  Total shape: {combined.shape}")
    return combined


# ─────────────────────────────────────────────
# STEP 2: CLEAN
# ─────────────────────────────────────────────
def clean_data(df):
    print("\n[2/6] Cleaning data...")
    initial_rows = len(df)

    # Fix inf and NaN
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    print(f"  Dropped {initial_rows - len(df):,} rows with inf/NaN")

    # Drop duplicate column
    if "Fwd Header Length.1" in df.columns:
        df.drop(columns=["Fwd Header Length.1"], inplace=True)
        print("  Dropped duplicate: Fwd Header Length.1")

    # Drop unlearnable classes
    before = len(df)
    df = df[~df["Label"].isin(DROP_CLASSES)]
    dropped = before - len(df)
    print(f"  Dropped {dropped:,} rows for unlearnable classes: {DROP_CLASSES}")

    # Normalize label encoding inconsistencies
    df["Label"] = df["Label"].str.strip()

    print(f"  Final shape after cleaning: {df.shape}")
    return df


# ─────────────────────────────────────────────
# STEP 3: FEATURE ENGINEERING & SELECTION
# ─────────────────────────────────────────────
def select_features(df):
    print("\n[3/6] Feature selection...")

    feature_cols = [c for c in df.columns if c != "Label"]
    X = df[feature_cols]
    y = df["Label"]

    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    print(f"  Classes: {list(le.classes_)}")

    # Quick RF to get feature importances (on a 20% sample to save time)
    print("  Running quick RF for feature importance (sampled)...")
    sample_size = min(100_000, len(X))
    idx = np.random.choice(len(X), sample_size, replace=False)
    X_sample = X.iloc[idx]
    y_sample = y_encoded[idx]

    rf_selector = RandomForestClassifier(
        n_estimators=50, max_depth=10,
        random_state=42, n_jobs=-1, class_weight="balanced"
    )
    rf_selector.fit(X_sample, y_sample)

    importances = pd.Series(
        rf_selector.feature_importances_, index=feature_cols
    ).sort_values(ascending=False)

    # Keep top 30 features
    top_features = importances.head(30).index.tolist()
    print(f"  Selected top 30 features:")
    for i, feat in enumerate(top_features):
        print(f"    [{i+1:2d}] {feat:<40} importance: {importances[feat]:.4f}")

    # Save feature importance plot
    plt.figure(figsize=(10, 8))
    importances.head(30).sort_values().plot(kind="barh")
    plt.title("Top 30 Feature Importances")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/feature_importances.png", dpi=100)
    plt.close()
    print(f"  Saved feature importance chart → {OUTPUT_DIR}/feature_importances.png")

    return X[top_features], y_encoded, le, top_features


# ─────────────────────────────────────────────
# STEP 4: TRAIN / TEST SPLIT + SMOTE
# ─────────────────────────────────────────────
def prepare_splits(X, y):
    print("\n[4/6] Splitting and balancing...")

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"  Train: {X_train.shape}, Test: {X_test.shape}")

    # Scale
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled  = scaler.transform(X_test)

    # Skip SMOTE — use class_weight in XGBoost instead
    # SMOTE on 2M+ rows takes too long and isn't necessary
    # XGBoost's class_weight handles imbalance effectively
    print("  Skipping SMOTE — using class_weight='balanced' in XGBoost instead.")
    print("  This is faster and works well with XGBoost on large datasets.")

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler

# ─────────────────────────────────────────────
# STEP 5: TRAIN XGBOOST
# ─────────────────────────────────────────────
def train_model(X_train, y_train):
    print("\n[5/6] Training XGBoost model...")

    model = XGBClassifier(
    n_estimators=200,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric="mlogloss",
    random_state=42,
    n_jobs=-1,
    tree_method="hist",   # changed from gpu_hist
    device="cuda"         # this handles GPU automatically in newer API
)
    model.fit(X_train, y_train, verbose=False)
    print("  Training complete.")
    return model

# ─────────────────────────────────────────────
# STEP 6: EVALUATE
# ─────────────────────────────────────────────
def evaluate_model(model, X_test, y_test, le):
    print("\n[6/6] Evaluating model...")
    y_pred = model.predict(X_test)

    # Decode labels for readable report
    y_test_labels = le.inverse_transform(y_test)
    y_pred_labels = le.inverse_transform(y_pred)

    print("\n=== CLASSIFICATION REPORT ===")
    print(classification_report(y_test_labels, y_pred_labels, zero_division=0))

    # Macro F1
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    print(f"Macro F1 Score: {macro_f1:.4f}")

    # Confusion Matrix
    classes = le.classes_
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(12, 9))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        xticklabels=classes, yticklabels=classes
    )
    plt.title("Confusion Matrix")
    plt.ylabel("Actual")
    plt.xlabel("Predicted")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/confusion_matrix.png", dpi=100)
    plt.close()
    print(f"  Saved confusion matrix → {OUTPUT_DIR}/confusion_matrix.png")

    return macro_f1


# ─────────────────────────────────────────────
# RISK SCORE FUNCTION (for FastAPI inference)
# ─────────────────────────────────────────────
def compute_risk_score(predicted_label: str, confidence: float) -> int:
    """
    Returns risk score 1-10 based on:
    - Base severity from RISK_MAP
    - Scaled by model confidence (0-1)

    Usage in FastAPI:
        label = le.inverse_transform([pred])[0]
        confidence = float(proba.max())
        score = compute_risk_score(label, confidence)
    """
    base = RISK_MAP.get(predicted_label, 5)   # default 5 if label unknown
    if predicted_label == "BENIGN":
        # Low confidence benign = slightly elevated score
        score = max(1, round(base * (1 - confidence) * 3))
    else:
        # High confidence attack = score close to base
        score = round(base * (0.5 + 0.5 * confidence))
    return int(np.clip(score, 1, 10))


# ─────────────────────────────────────────────
# MAIN
# ─────────────────────────────────────────────
if __name__ == "__main__":
    print("=" * 60)
    print("  Neural-Trace ML Pipeline")
    print("=" * 60)

    df = load_data(DATA_DIR)
    df = clean_data(df)
    X, y, le, top_features = select_features(df)
    X_train, X_test, y_train, y_test, scaler = prepare_splits(X, y)
    model = train_model(X_train, y_train)
    macro_f1 = evaluate_model(model, X_test, y_test, le)

    # Save artifacts
    joblib.dump(model,        MODEL_PATH)
    joblib.dump(scaler,       SCALER_PATH)
    joblib.dump(le,           ENCODER_PATH)
    joblib.dump(top_features, FEATURES_PATH)

    print(f"\n✓ Model saved     → {MODEL_PATH}")
    print(f"✓ Scaler saved    → {SCALER_PATH}")
    print(f"✓ Encoder saved   → {ENCODER_PATH}")
    print(f"✓ Features saved  → {FEATURES_PATH}")
    print(f"\nFinal Macro F1: {macro_f1:.4f}")
    print("\nPipeline complete.")

  Neural-Trace ML Pipeline

[1/6] Loading CSVs...
  Loading: Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
  Loading: Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
  Loading: Friday-WorkingHours-Morning.pcap_ISCX.csv
  Loading: Monday-WorkingHours.pcap_ISCX.csv
  Loading: Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
  Loading: Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
  Loading: Tuesday-WorkingHours.pcap_ISCX.csv
  Loading: Wednesday-workingHours.pcap_ISCX.csv
  Total shape: (2830743, 79)

[2/6] Cleaning data...
  Dropped 2,867 rows with inf/NaN
  Dropped duplicate: Fwd Header Length.1
  Dropped 68 rows for unlearnable classes: ['Infiltration', 'Heartbleed', 'Web Attack � Sql Injection']
  Final shape after cleaning: (2827808, 78)

[3/6] Feature selection...
  Classes: ['BENIGN', 'Bot', 'DDoS', 'DoS GoldenEye', 'DoS Hulk', 'DoS Slowhttptest', 'DoS slowloris', 'FTP-Patator', 'PortScan', 'SSH-Patator', 'Web Attack � Brute Force', 'Web Attack � XSS']


In [6]:
from google.colab import files

files.download('./model_output/model.pkl')
files.download('./model_output/scaler.pkl')
files.download('./model_output/label_encoder.pkl')
files.download('./model_output/feature_list.pkl')
files.download('./model_output/confusion_matrix.png')
files.download('./model_output/feature_importances.png')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [7]:
import shutil
shutil.copytree('./model_output', '/content/drive/MyDrive/NeuralTrace_ModelOutput')
print("Backed up to Drive.")

Backed up to Drive.


In [8]:
import joblib

features = joblib.load('./model_output/feature_list.pkl')
print(f"Total features: {len(features)}")
print("\nFeature list:")
for i, f in enumerate(features):
    print(f"  [{i+1}] {f}")

Total features: 30

Feature list:
  [1] Destination Port
  [2] Init_Win_bytes_backward
  [3] Init_Win_bytes_forward
  [4] Packet Length Mean
  [5] Bwd Packets/s
  [6] Flow IAT Std
  [7] Flow Packets/s
  [8] min_seg_size_forward
  [9] Flow Duration
  [10] Fwd IAT Mean
  [11] Subflow Fwd Bytes
  [12] Fwd Packet Length Max
  [13] Bwd Header Length
  [14] Fwd Packets/s
  [15] Flow IAT Max
  [16] Flow IAT Mean
  [17] Total Length of Fwd Packets
  [18] Fwd IAT Total
  [19] Fwd IAT Min
  [20] Fwd IAT Max
  [21] Average Packet Size
  [22] Flow Bytes/s
  [23] Fwd IAT Std
  [24] Bwd Packet Length Mean
  [25] Subflow Bwd Bytes
  [26] Avg Fwd Segment Size
  [27] Avg Bwd Segment Size
  [28] Packet Length Variance
  [29] Bwd Packet Length Min
  [30] Packet Length Std
